# 04 — Severity Benchmark: Classical CV vs Fine-tuned CNN

Both severity methods are evaluated on the held-out **test set**.

| Method | Description |
|---|---|
| Classical CV | Edge density (Canny) + depth score (center darkness) + texture score |
| Fine-tuned CNN | YOLOv8n-cls trained on auto-labelled severity crops |

The winner is written to `configs/pipeline_config.yaml` and used at runtime.

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

BASE_DIR = Path('..').resolve()
SEVERITY_TEST = BASE_DIR / 'data' / 'processed' / 'severity_crops' / 'test'
MODEL_PATH    = BASE_DIR / 'models' / 'severity_model.pt'

sys.path.insert(0, str(BASE_DIR / 'src'))
from analyzers.pothole_analyzer import _compute_cv_scores, _cv_severity

CLASSES = ['Low', 'Medium', 'High']
CNN_IDX_TO_LABEL = {0: 'High', 1: 'Low', 2: 'Medium'}  # alphabetical order

## 1. Load Test Set

In [ ]:
test_items = []  # (image_bgr, true_label, path)
for cls in CLASSES:
    for p in (SEVERITY_TEST / cls).glob('*'):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
        img = cv2.imread(str(p))
        if img is not None:
            test_items.append((img, cls, p))

y_true = [item[1] for item in test_items]
print(f'Test set: {len(test_items)} crops')
from collections import Counter
print('Distribution:', dict(Counter(y_true)))

## 2. Classical CV Predictions

In [ ]:
cv_preds  = []
cv_scores_list = []

for img, label, _ in test_items:
    scores = _compute_cv_scores(img)
    cv_scores_list.append(scores)
    cv_preds.append(_cv_severity(scores))

cv_acc = sum(t == p for t, p in zip(y_true, cv_preds)) / len(y_true)
print(f'Classical CV Accuracy: {cv_acc:.4f} ({cv_acc*100:.1f}%)')

## 3. CNN Predictions

In [ ]:
cnn_preds = None
cnn_acc   = None

if MODEL_PATH.exists():
    from ultralytics import YOLO
    cnn_model = YOLO(str(MODEL_PATH))
    cnn_preds = []
    for img, label, _ in test_items:
        result   = cnn_model(img, imgsz=128, verbose=False)
        top1_idx = int(result[0].probs.top1)
        cnn_preds.append(CNN_IDX_TO_LABEL.get(top1_idx, 'Low'))
    cnn_acc = sum(t == p for t, p in zip(y_true, cnn_preds)) / len(y_true)
    print(f'CNN Accuracy: {cnn_acc:.4f} ({cnn_acc*100:.1f}%)')
else:
    print(f'CNN model not found at {MODEL_PATH}')

## 4. Accuracy Comparison

In [ ]:
methods = ['Classical CV']
accs    = [cv_acc]
colors  = ['#3498db']

if cnn_acc is not None:
    methods.append('Fine-tuned CNN')
    accs.append(cnn_acc)
    colors.append('#e74c3c')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(methods, accs, color=colors, width=0.4)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.2%}', ha='center', fontsize=12, fontweight='bold')

winner_idx = accs.index(max(accs))
bars[winner_idx].set_edgecolor('gold')
bars[winner_idx].set_linewidth(3)

ax.set_ylim(0, 1.1)
ax.set_title('Severity Classification Accuracy — CV vs CNN', fontsize=13)
ax.set_ylabel('Accuracy')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
ax.legend()
plt.tight_layout()
plt.show()

print(f'WINNER: {methods[winner_idx]} ({accs[winner_idx]:.2%})')

## 5. Side-by-Side Confusion Matrices

In [ ]:
n_methods = 1 + (1 if cnn_preds else 0)
fig, axes = plt.subplots(1, n_methods, figsize=(6 * n_methods, 5))
if n_methods == 1: axes = [axes]

for ax, (preds, title) in zip(axes, [
    (cv_preds, f'Classical CV ({cv_acc:.2%})'),
    *( [(cnn_preds, f'Fine-tuned CNN ({cnn_acc:.2%})')] if cnn_preds else [] )
]):
    cm   = confusion_matrix(y_true, preds, labels=CLASSES)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12)

plt.suptitle('Confusion Matrices — Severity Benchmark', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Per-Class F1 Comparison

In [ ]:
from sklearn.metrics import f1_score

cv_f1  = f1_score(y_true, cv_preds,  labels=CLASSES, average=None)

x      = np.arange(len(CLASSES))
width  = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, cv_f1, width, label='Classical CV', color='#3498db')

if cnn_preds:
    cnn_f1 = f1_score(y_true, cnn_preds, labels=CLASSES, average=None)
    ax.bar(x + width/2, cnn_f1, width, label='Fine-tuned CNN', color='#e74c3c')

ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylim(0, 1.1); ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 — CV vs CNN')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Side-by-Side Examples: Correct vs Incorrect

In [ ]:
# Find cases where CV is wrong but CNN is right, and vice versa
if cnn_preds:
    cv_wrong_cnn_right  = [(i, test_items[i]) for i in range(len(y_true))
                           if cv_preds[i] != y_true[i] and cnn_preds[i] == y_true[i]][:4]
    cv_right_cnn_wrong  = [(i, test_items[i]) for i in range(len(y_true))
                           if cv_preds[i] == y_true[i] and cnn_preds[i] != y_true[i]][:4]

    for cases, title in [
        (cv_wrong_cnn_right, 'CNN correct, CV wrong'),
        (cv_right_cnn_wrong, 'CV correct, CNN wrong'),
    ]:
        if not cases: print(f'No {title} cases found.'); continue
        fig, axes = plt.subplots(1, len(cases), figsize=(3*len(cases), 3))
        if len(cases) == 1: axes = [axes]
        for ax, (idx, (img, true, path)) in zip(axes, cases):
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis('off')
            ax.set_title(f'True:{true}\nCV:{cv_preds[idx]}\nCNN:{cnn_preds[idx]}',
                         fontsize=8)
        plt.suptitle(title, fontsize=11)
        plt.tight_layout()
        plt.show()
else:
    print('CNN model not available for comparison examples.')

## 8. Run Full Benchmark Script & Write Config

In [ ]:
# This writes the winner to configs/pipeline_config.yaml
import subprocess, sys
result = subprocess.run([sys.executable, '../src/benchmark_severity.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

import yaml
cfg = yaml.safe_load(open('../configs/pipeline_config.yaml'))
print(f'\nPipeline config written:')
for k, v in cfg.items(): print(f'  {k}: {v}')